In [ ]:
import json
import os
from PIL import Image
import torch
# Đường dẫn
json_path = "/mnt/VLAI_data/ViVQA-X/ViVQA-X_train.json"
coco_img_dir = "/mnt/VLAI_data/COCO_Images/train2014/"

# Đọc file JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

samples = []
for item in data:
    img_path = os.path.join(coco_img_dir, item["image_name"])
    # Mở ảnh thành PIL Image
    image = Image.open(img_path).convert("RGB")
    sample = {
        "question": item["question"],
        "image": image,  # <-- PIL Image object
        "image_path": img_path,
        "explanation": item["explanation"],  # list
        "answer": item["answer"],
        "question_id": item["question_id"],
        "question_type": item["question_type"]
    }
    samples.append(sample)



In [10]:
import json
import os
from PIL import Image
import torch
from collections import defaultdict

# Đường dẫn
json_path = "/mnt/VLAI_data/ViVQA-X/ViVQA-X_train.json"
coco_img_dir = "/mnt/VLAI_data/COCO_Images/train2014/"

# Đọc file JSON
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Tạo samples
samples = []
for item in data:
    img_path = os.path.join(coco_img_dir, item["image_name"])
    image = Image.open(img_path).convert("RGB")
    sample = {
        "question": item["question"],
        "image": image,
        "image_path": img_path,
        "explanation": item["explanation"],
        "answer": item["answer"],
        "question_id": item["question_id"],
        "question_type": item["question_type"]
    }
    samples.append(sample)

# Nhóm samples theo question_type
samples_by_type = defaultdict(list)
for sample in samples:
    samples_by_type[sample["question_type"]].append(sample)

# In 10 sample cho mỗi question_type
sorted_types = sorted(samples_by_type.keys())

for q_type in sorted_types:
    type_samples = samples_by_type[q_type]
    print(f"\n{'='*80}")
    print(f"QUESTION TYPE: {q_type} (Total: {len(type_samples)} samples)")
    print('='*80)

    # In 10 samples
    for idx in range(min(20, len(type_samples))):
        sample = type_samples[idx]
        print(f"\n[Sample {idx + 1}/20]")
        print(f"Question type: {sample['question_type']}")
        print(f"Question: {sample['question']}")
        print(f"Answer: {sample['answer']}")
        print(f"Explanation: {sample['explanation']}")


QUESTION TYPE: are (Total: 14 samples)

[Sample 1/20]
Question type: are
Question: Tất cả các loại trái cây được trưng bày có phải là táo không?
Answer: đúng
Explanation: ['chỉ có một dãy táo được bày trong giỏ trưng bày bằng liễu gai']

[Sample 2/20]
Question type: are
Question: Có phải tất cả các phông chữ đều giống nhau không?
Answer: không
Explanation: ["từ 'tình yêu' xuất hiện bằng phông chữ rất phức tạp, trong đó mỗi chữ cái xuất hiện trong một hình vuông và phần còn lại của câu trích dẫn xuất hiện bằng phông chữ truyền thống hơn"]

[Sample 3/20]
Question type: are
Question: Xe đạp có hướng về bên trái không?
Answer: không
Explanation: ['những chiếc xe đạp được xếp hàng với bánh xe hướng về bên phải']

[Sample 4/20]
Question type: are
Question: Có phải đó là những bình đựng xà phòng lỏng?
Answer: đúng
Explanation: ['chúng có hình dạng giống như chúng sẽ phân phối xà phòng lỏng']

[Sample 5/20]
Question type: are
Question: Xe đạp có hướng về bên trái không?
Answer: đúng
Explanati

In [ ]:
# are , are the, are these --> ưu tiên đếm danh từ. Có thể gom về are

# does this (type other), is this (100%), does the (type other), is this a (type other), is the (type other), is it (type other)
# --> những question type có  đảo động từ tobe hoặc trợ động từ lên trước + N và nó thuộc answer type other --> multiple choice

In [9]:
import json
import os
from PIL import Image
import torch
from collections import defaultdict, Counter
from underthesea import word_tokenize, pos_tag
import matplotlib.pyplot as plt
import numpy as np

# ============================================================================
# BƯỚC 1: ĐỌC DỮ LIỆU
# ============================================================================
json_path = "/mnt/VLAI_data/ViVQA-X/ViVQA-X_train.json"
coco_img_dir = "/mnt/VLAI_data/COCO_Images/train2014/"

print("Đang đọc dữ liệu...")
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# Tạo samples
samples = []
for item in data:
    img_path = os.path.join(coco_img_dir, item["image_name"])
    image = Image.open(img_path).convert("RGB")
    sample = {
        "question": item["question"],
        "image": image,
        "image_path": img_path,
        "explanation": item["explanation"],
        "answer": item["answer"],
        "question_id": item["question_id"],
        "question_type": item["question_type"]
    }
    samples.append(sample)

print(f"✓ Đã load {len(samples)} samples")

# Nhóm samples theo question_type
samples_by_type = defaultdict(list)
for sample in samples:
    samples_by_type[sample["question_type"]].append(sample)

print(f"✓ Có {len(samples_by_type)} question types\n")

# ============================================================================
# BƯỚC 2: POS TAGGING CHO TOÀN BỘ DATASET
# ============================================================================
sorted_types = sorted(samples_by_type.keys())
pos_stats_by_type = {}

print("="*80)
print("ĐANG PHÂN TÍCH POS TAGGING CHO TOÀN BỘ DATASET")
print("="*80)

for idx, q_type in enumerate(sorted_types, 1):
    type_samples = samples_by_type[q_type]
    print(f"[{idx}/{len(sorted_types)}] Đang xử lý: {q_type} ({len(type_samples)} samples)...", end=" ")

    # Counter để đếm POS tags
    pos_counter = Counter()

    # Phân tích TẤT CẢ samples trong question_type này
    for sample in type_samples:
        question = sample['question']

        # POS Tagging
        pos_tags = pos_tag(question)

        # Đếm các POS tags
        for word, tag in pos_tags:
            pos_counter[tag] += 1

    # Lưu thống kê
    pos_stats_by_type[q_type] = pos_counter
    print("✓")

print("\n" + "="*80)
print("✓ HOÀN THÀNH PHÂN TÍCH!")
print("="*80)

# ============================================================================
# BƯỚC 3: TỔNG HỢP THỐNG KÊ
# ============================================================================
print("\n" + "="*80)
print("THỐNG KÊ POS TAGS THEO QUESTION_TYPE")
print("="*80)

for q_type in sorted_types:
    pos_counter = pos_stats_by_type[q_type]
    total_tags = sum(pos_counter.values())

    print(f"\n{q_type}:")
    print(f"  Total samples: {len(samples_by_type[q_type])}")
    print(f"  Total tags: {total_tags}")
    print(f"  Top 5 POS tags:")
    for tag, count in pos_counter.most_common(5):
        percentage = (count / total_tags) * 100
        print(f"    {tag:<10} {count:>6} ({percentage:>5.2f}%)")

# ============================================================================
# BƯỚC 4: VISUALIZATION VỚI MATPLOTLIB
# ============================================================================
print("\n" + "="*80)
print("TẠO BIỂU ĐỒ VISUALIZATION")
print("="*80)

# Lấy tất cả unique POS tags
all_tags = set()
for pos_counter in pos_stats_by_type.values():
    all_tags.update(pos_counter.keys())
all_tags = sorted(all_tags)

print(f"Tổng số POS tags khác nhau: {len(all_tags)}")
print(f"POS tags: {all_tags}")

# Tạo ma trận dữ liệu
matrix_data = []
for q_type in sorted_types:
    row = [pos_stats_by_type[q_type].get(tag, 0) for tag in all_tags]
    matrix_data.append(row)

matrix_data = np.array(matrix_data)

# ============================================================================
# BIỂU ĐỒ 1: HEATMAP - POS TAGS DISTRIBUTION
# ============================================================================
fig1, ax1 = plt.subplots(figsize=(14, 10))
im = ax1.imshow(matrix_data, cmap='YlOrRd', aspect='auto')

# Thiết lập ticks
ax1.set_xticks(np.arange(len(all_tags)))
ax1.set_yticks(np.arange(len(sorted_types)))
ax1.set_xticklabels(all_tags, fontsize=10)
ax1.set_yticks(np.arange(len(sorted_types)))
ax1.set_yticklabels(sorted_types, fontsize=9)

# Xoay labels
plt.setp(ax1.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Thêm colorbar
cbar = plt.colorbar(im, ax=ax1)
cbar.set_label('Frequency', rotation=270, labelpad=20, fontsize=12)

# Thêm giá trị vào cells
for i in range(len(sorted_types)):
    for j in range(len(all_tags)):
        if matrix_data[i, j] > 0:
            text = ax1.text(j, i, int(matrix_data[i, j]),
                          ha="center", va="center", color="black", fontsize=7)

ax1.set_title("POS Tags Distribution across Question Types\n(All Dataset)", 
             fontsize=14, fontweight='bold', pad=20)
ax1.set_xlabel("POS Tags", fontsize=12, fontweight='bold')
ax1.set_ylabel("Question Types", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('pos_heatmap_all_dataset.png', dpi=300, bbox_inches='tight')
print("✓ Đã lưu: pos_heatmap_all_dataset.png")
plt.close()

# ============================================================================
# BIỂU ĐỒ 2: BAR CHART - TOP POS TAGS PER QUESTION TYPE
# ============================================================================
fig2, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for idx, q_type in enumerate(sorted_types[:9]):  # Hiển thị 9 question types đầu
    pos_counter = pos_stats_by_type[q_type]
    top_tags = pos_counter.most_common(10)

    if top_tags:
        tags = [tag for tag, _ in top_tags]
        counts = [count for _, count in top_tags]

        axes[idx].bar(tags, counts, color='steelblue', alpha=0.8)
        axes[idx].set_title(f'{q_type}\n({len(samples_by_type[q_type])} samples)', 
                           fontsize=11, fontweight='bold')
        axes[idx].set_xlabel('POS Tags', fontsize=9)
        axes[idx].set_ylabel('Frequency', fontsize=9)
        axes[idx].tick_params(axis='x', rotation=45, labelsize=8)
        axes[idx].tick_params(axis='y', labelsize=8)
        axes[idx].grid(axis='y', alpha=0.3)

# Ẩn các subplot trống nếu có
for idx in range(len(sorted_types), 9):
    axes[idx].axis('off')

plt.suptitle('Top 10 POS Tags per Question Type (All Dataset)', 
            fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('pos_bar_charts_all_dataset.png', dpi=300, bbox_inches='tight')
print("✓ Đã lưu: pos_bar_charts_all_dataset.png")
plt.close()

# ============================================================================
# BIỂU ĐỒ 3: STACKED BAR CHART - POS DISTRIBUTION COMPARISON
# ============================================================================
fig3, ax3 = plt.subplots(figsize=(16, 8))

# Tính phần trăm cho mỗi question type
matrix_percentage = []
for q_type in sorted_types:
    pos_counter = pos_stats_by_type[q_type]
    total = sum(pos_counter.values())
    row = [(pos_counter.get(tag, 0) / total * 100) if total > 0 else 0 for tag in all_tags]
    matrix_percentage.append(row)

matrix_percentage = np.array(matrix_percentage)

# Tạo stacked bar chart
x = np.arange(len(sorted_types))
bottom = np.zeros(len(sorted_types))
colors = plt.cm.Set3(np.linspace(0, 1, len(all_tags)))

for idx, tag in enumerate(all_tags):
    values = matrix_percentage[:, idx]
    ax3.bar(x, values, bottom=bottom, label=tag, color=colors[idx], alpha=0.8)
    bottom += values

ax3.set_xlabel('Question Types', fontsize=12, fontweight='bold')
ax3.set_ylabel('Percentage (%)', fontsize=12, fontweight='bold')
ax3.set_title('POS Tags Distribution (Percentage) across Question Types\n(All Dataset)', 
             fontsize=14, fontweight='bold', pad=20)
ax3.set_xticks(x)
ax3.set_xticklabels(sorted_types, rotation=45, ha='right', fontsize=9)
ax3.legend(title='POS Tags', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('pos_stacked_bar_all_dataset.png', dpi=300, bbox_inches='tight')
print("✓ Đã lưu: pos_stacked_bar_all_dataset.png")
plt.close()

# ============================================================================
# BIỂU ĐỒ 4: PIE CHART - OVERALL POS DISTRIBUTION
# ============================================================================
fig4, ax4 = plt.subplots(figsize=(12, 10))

# Tổng hợp tất cả POS tags
overall_counter = Counter()
for pos_counter in pos_stats_by_type.values():
    overall_counter.update(pos_counter)

# Lấy top POS tags
top_overall = overall_counter.most_common(10)
labels = [tag for tag, _ in top_overall]
sizes = [count for _, count in top_overall]

# Tạo pie chart
colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
explode = [0.05 if i == 0 else 0 for i in range(len(labels))]

wedges, texts, autotexts = ax4.pie(sizes, labels=labels, autopct='%1.1f%%',
                                     startangle=90, colors=colors, explode=explode,
                                     textprops={'fontsize': 11})

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

ax4.set_title('Overall POS Tags Distribution\n(Top 10 - All Dataset)', 
             fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('pos_pie_chart_all_dataset.png', dpi=300, bbox_inches='tight')
print("✓ Đã lưu: pos_pie_chart_all_dataset.png")
plt.close()

# ============================================================================
# LƯU THỐNG KÊ RA FILE CSV
# ============================================================================
import csv

# Lưu ma trận POS tags
with open('pos_statistics_all_dataset.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f)

    # Header
    header = ['Question_Type', 'Total_Samples', 'Total_Tags'] + all_tags
    writer.writerow(header)

    # Data
    for q_type in sorted_types:
        pos_counter = pos_stats_by_type[q_type]
        total_samples = len(samples_by_type[q_type])
        total_tags = sum(pos_counter.values())
        row = [q_type, total_samples, total_tags]
        row.extend([pos_counter.get(tag, 0) for tag in all_tags])
        writer.writerow(row)

print("✓ Đã lưu: pos_statistics_all_dataset.csv")

print("\n" + "="*80)
print("✓ HOÀN THÀNH TẤT CẢ!")
print("="*80)
print("\nĐã tạo 4 biểu đồ:")
print("  1. pos_heatmap_all_dataset.png - Heatmap phân bố POS tags")
print("  2. pos_bar_charts_all_dataset.png - Bar charts cho từng question type")
print("  3. pos_stacked_bar_all_dataset.png - Stacked bar chart so sánh")
print("  4. pos_pie_chart_all_dataset.png - Pie chart tổng quan")
print("\nVà 1 file CSV:")
print("  - pos_statistics_all_dataset.csv - Thống kê chi tiết")
print("="*80)

/opt/miniconda3/envs/project_vivqanle_grpo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đang đọc dữ liệu...
✓ Đã load 29459 samples
✓ Có 53 question types

ĐANG PHÂN TÍCH POS TAGGING CHO TOÀN BỘ DATASET
[1/53] Đang xử lý: are (14 samples)... ✓
[2/53] Đang xử lý: are the (169 samples)... ✓
[3/53] Đang xử lý: are there (3 samples)... ✓
[4/53] Đang xử lý: are these (80 samples)... ✓
[5/53] Đang xử lý: are they (57 samples)... ✓
[6/53] Đang xử lý: can you (53 samples)... ✓
[7/53] Đang xử lý: could (24 samples)... ✓
[8/53] Đang xử lý: do (20 samples)... ✓
[9/53] Đang xử lý: do you (5 samples)... ✓
[10/53] Đang xử lý: does the (2446 samples)... ✓
[11/53] Đang xử lý: does this (1645 samples)... ✓
[12/53] Đang xử lý: has (31 samples)... ✓
[13/53] Đang xử lý: how (16 samples)... ✓
[14/53] Đang xử lý: is (409 samples)... ✓
[15/53] Đang xử lý: is he (174 samples)... ✓
[16/53] Đang xử lý: is it (1596 samples)... ✓
[17/53] Đang xử lý: is that a (131 samples)... ✓
[18/53] Đang xử lý: is the (3832 samples)... ✓
[19/53] Đang xử lý: is the man (369 samples)... ✓
[20/53] Đang xử lý: is the